<p align="center">
<a href="https://duckietown.com"><img src="../../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

# 🚙 💻 Wheel encoders

Encoders are sensors that convert (analog) angular position, or motion of a shaft, into a digital signal. 

In Duckietown we use Hall effect encoders [link to Wikipedia](https://en.wikipedia.org/wiki/Hall-effect_sensor), which extract the _incremental_ change in angular position of the wheels. Every time the shaft rotates a certain set angle, i.e., the resolution of the encoder, it emits a pulse. We call these pulses "ticks".  

We can use ticks from both wheels to measure the change to the position of the Duckiebot while it moves. 

In this activity we learn how to access the data coming from the wheel encoders of our Duckiebot, and understand what each field means. We will use this data for later activities.

## 🚙 💻 Obtaining wheel encoder measurements. 

Let's look at the raw data that the wheel encoders produce. 

### 💻 Read data from wheel encoders

Follow the procedure in [the README](../../README.md) to start your (real or virtual) robot and its ROS 2 stack.

The code editor you are reading this from can talk to your robot directly. Open a terminal inside the code editor (menu > Terminal > New Terminal) and type:

       ros2 topic echo /[ROBOT_NAME]/left_wheel_encoder_driver_node/tick

where `[ROBOT_NAME]` could be a virtual or a real robot. 

**Note**: run `dt-ros2-connect [ROBOT_NAME]` in the terminal first so the `ros2` tools talk to your robot. You can list all the topics published by your robot with `ros2 topic list` (the graph takes a few seconds to sync through the robot's zenoh router: if the list comes back empty, use `ros2 topic list --no-daemon --spin-time 5`).

Now make your robot move (push it gently by hand, or drive it with the virtual joystick below) and you will start seeing wheel encoder messages streaming on your terminal.

Messages will look like this:

```    
header:
  stamp:
    sec: 1618436796
    nanosec: 55785179
  frame_id: [ROBOT_NAME]/left_wheel_axis
data: 4
resolution: 135
type: 1
---
```    

Let's look at what each field means:

* `stamp`: the timestamp of the message (seconds and nanoseconds since the Unix epoch). 
* `frame_id`: the name of the reference frame the measurement refers to (the axis of the left wheel here).
* `data`: is the cumulative count of ticks from the encoder in this instance. It will increase if the wheel is spinning forward, decrease if backwards. This is the actual measurement we can use to build our algorithms going forward.
* `resolution`: is the total number of ticks for each full revolution of the wheel (a constant).
* `type`: indicates the kind of encoder measurements. `1` stands for [incremental measurements](https://github.com/duckietown/dt-ros2-commons/blob/ente/packages/duckietown_msgs/msg/WheelEncoderStamped.msg). 

### 💻 Read data from wheel encoders in this notebook

The notebooks can talk to the robot too. Fill in the name of your robot and run the next cells: the first one receives one encoder message and prints it, the second one plots the tick count of both wheels live while you drive the robot with the virtual joystick.


In [ ]:
from lx_ros2_tools.session import RosSession
from lx_ros2_tools.panel import ActionButtons, ImageView, TopicPlot, TrajectoryPlot
from lx_kinematics_tools import Joystick

ROBOT = "ROBOT_NAME"   # <-- the name of your (real or virtual) robot
ros = RosSession(ROBOT)   # connects to the robot (its ROS 2 topics are now reachable from this notebook)


In [ ]:
from duckietown_msgs.msg import WheelEncoderStamped

# wait for one message from the left wheel encoder and print it (make the wheel turn if nothing arrives)
encoder_msg = ros.wait_for_message("left_wheel_encoder_driver_node/tick", WheelEncoderStamped, timeout=10.0, sensor=True)
print(encoder_msg)


In [ ]:
# live tick count of both wheels, and a virtual joystick to drive the robot
TopicPlot(ros, "left_wheel_encoder_driver_node/tick", "duckietown_msgs/msg/WheelEncoderStamped", ["data"], sensor=True, title="left wheel ticks")
TopicPlot(ros, "right_wheel_encoder_driver_node/tick", "duckietown_msgs/msg/WheelEncoderStamped", ["data"], sensor=True, title="right wheel ticks")
Joystick(ros)   # if the robot does not move: Joystick(ros, topic="car_cmd") (older robot images, see the README)


## Reading the number of ticks from each wheel

The wheel encoder message above provides several pieces of information. Let's extract data from it. 


In [ ]:
import sys
sys.path.append('../../tests')
from unit_test import UnitTestMessage

# We define this function just to show how to extract data from the encoder_msg, which is the data coming from the wheel encoders

def EncoderCallback(encoder_msg):
    
    N_tot = encoder_msg.resolution # number of ticks per wheel revolution
    ticks = encoder_msg.data # incremental count of ticks from the encoder
    
    #Let's see if we've done it right
    print("The received message is :")
    print()
    print(encoder_msg)
    print()
    print(f"N. of ticks : {ticks}")
    print(f"Total ticks : {N_tot}")


# Testing the callback
UnitTestMessage(EncoderCallback)


You should now be able to read and understand wheel encoder data, and extract fields of interest from the messages. You can proceed to the next tutorial, the [odometry activity](../03-Odometry/odometry.ipynb).